In [7]:
import importlib as il
import numpy as np
import more_itertools as mit

import gurobipy as gp

import gurobi_utils as gu
import dikin_utils as du
import plot_utils as pu

import example_loader as el
import miplib_loader as ml
import jsplib_loader as jl

%matplotlib inline
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

<gurobipy.Env, Parameter changes: WLSAccessID=(user-defined), WLSSecret=(user-defined), LicenseID=2586148, OutputFlag=0>

What we want to do here:
1. Find an LP optimum for some problem.
2. Back away from that optimum into the interior sqrt(n)/2 distance.
3. Find the eigenvectors for Dikin's H at that point.
4. Round those to be integer values.
5. Run the LLL reduction on that result.
6. Make that be unimodular if it's not.
7. Transform the original problem and run the MIP solver for it.

In [10]:
def retreat_from_optimum_via_average_vector(V: np.ndarray, x: np.ndarray):
    """Retreat from the optimum by moving in the direction of the average vector."""
    avg = np.mean(V, axis=0)
    nrm = np.linalg.norm(avg)
    distance = np.sqrt(len(x)) * 0.5
    return x - avg * distance / nrm

def find_corner(relaxed: gp.Model, int_vars, int_var_idx, distance):
    basis = gu.read_basis(relaxed)
    tableau, col_to_var_idx, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=0)
    negated_vars = [basis[nr] for nr in negated_rows]

    # the current understanding (from nlhdlr_quadratic.c in SCIP): 
    # negate all columns with variables at status -1
    # and negate all columns match slack variables of type <
    # for col, j in enumerate(col_to_var_idx):
    #     if j < len(variables):
    #         print("Var INFO:", variables[j].VarName, "VBasis", variables[j].VBasis, "LB", variables[j].LB, "UB", variables[j].UB)
    #         # if variables[j].VBasis == -2:
    #             # tableau[:, col] = 1.0 - tableau[:, col]
    #         if variables[j].VBasis == -1:  # not sure what to do with VBasis=-3
    #             tableau[:, col] = -tableau[:, col]  # variables[j].LB
    #             if variables[j].LB != 0.0:
    #                 print("Warning: LB is nonzero for variable", variables[j].VarName, "LB", variables[j].LB, "UB", variables[j].UB)
    #     else:
    #         constraint = constraints[j - len(variables)]
    #         # this might not be right: scip has status and tests for A_i*x being at lower or upper bound
    #         # if np.isclose(constraint.Slack, 0.0, atol=tol):
    #         #     tableau[:, col] = -tableau[:, col]
    #         if constraint.Sense != '>':  # Achterberg said lt and lte are standard; should just need to flip gt
    #             tableau[:, col] = -tableau[:, col]

        # drop the rows of non-integer variables:
    to_drop = [i for i, b in enumerate(basis) if b not in int_var_idx]
    tableau = np.delete(tableau, to_drop, axis=0)  # TODO: don't even bother to read them in
    basis = np.delete(basis, to_drop) # update basis to match tableau
    
    mit.sort_together([basis, tableau], key_list=[0], key=int_var_idx.get)

    # TODO: get Hildebrand's opinion on this: we should only be considering integer variables?
    return tableau, int_vars.X


In [11]:
il.reload(el)
il.reload(gu)

instances = el.get_instances(env)
for name, instance in instances.items():
    model = instance.as_gurobi_model()
    int_vars, int_var_idx = gu.relax_int_or_bin_to_continuous(model)
    model.optimize()
    V, x = find_corner(model, int_vars, int_var_idx, np.sqrt(int_vars.size) / 4)
    x2 = retreat_from_optimum_via_average_vector(V, x)
    # x, y = make_x_y(model)
    # x2 = du.reverse_interior_point_gpt(model.getA(), model.getAttr("RHS"), x, y)
    print(name, ": Optimum:", x, "Retreat to:", x2)




   Relaxed 2 variables on 2D from bottom
2Dbelow : Optimum: [1.22222222 2.33333333] Retreat to: [1.10597458 1.6358475 ]
   Relaxed 2 variables on 2D no easy cut from bottom
2Dnoeasy : Optimum: [3.38297872 2.72340426] Retreat to: [2.686457   2.60151295]
   Relaxed 2 variables on 2D from top
2Dabove : Optimum: [1.22222222 2.33333333] Retreat to: [1.10597458 1.6358475 ]
   Relaxed 2 variables on 2D from bottom (upper bounded x)
2DbelowUBx : Optimum: [1.         2.11111111] Retreat to: [1.4730295  1.58552278]
   Relaxed 2 variables on 2D from bottom (steep, y<=2)
2DbelowUBy : Optimum: [1.05 2.  ] Retreat to: [1.25318564 2.67728546]
   Relaxed 2 variables on 2D slanted
2Dslant : Optimum: [4.25 2.5 ] Retreat to: [3.69109216 2.06684642]
   Relaxed 3 variables on Book example 6.3
Book_6_3 : Optimum: [0.5 0.5 1. ] Retreat to: [-0.3660254  0.5        1.       ]
   Relaxed 2 variables on 2D from bottom (manual slack on 2)


ValueError: operands could not be broadcast together with shapes (2,) (3,) 